5 个真实数据集：
HLN-A1
HLN-D1
E18.5
S2-E15
S2-E18

每个数据集：
50 epochs

training seeds = 0~9

warm-up = 10

BSRR spatial_k = 3

KMeans n_init = 20

KMeans random_state = 0

## Cell 1：检查正式实验 Kaggle 环境

In [1]:
import sys
import torch
import numpy as np
import sklearn
import yaml

print("Python:", sys.version)
print("Python executable:", sys.executable)
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("没有检测到 GPU，请先开启 Kaggle GPU。")

print("\nPASS: 基础环境正常。")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Python executable: /usr/bin/python3
PyTorch: 2.10.0+cu128
NumPy: 2.0.2
scikit-learn: 1.6.1
CUDA available: True
GPU: Tesla T4

PASS: 基础环境正常。


## Cell 2：安装并检查 anndata

In [2]:
import sys
import subprocess
import importlib.util

if importlib.util.find_spec("anndata") is None:
    print("Installing anndata...")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--no-deps",
            "anndata==0.11.4",
        ],
        check=True,
    )

import anndata

print("anndata:", anndata.__version__)
print("NumPy:", __import__("numpy").__version__)

print("\nPASS: anndata 正常。")

Installing anndata...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 4.2 MB/s eta 0:00:00
anndata: 0.11.4
NumPy: 2.0.2

PASS: anndata 正常。


## Cell 3：重新克隆最新 SpaMGCL 正式代码

In [3]:
from pathlib import Path
import shutil
import subprocess

repo_root = Path("/kaggle/working/SpaMGCL")
project_root = repo_root / "SpaMGCL"

if repo_root.exists():
    shutil.rmtree(repo_root)

subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "--branch",
        "main",
        "https://github.com/huqian122/SpaMGCL.git",
        str(repo_root),
    ],
    check=True,
)

assert project_root.exists()

print("Project root:")
print(project_root)

print("\nCommit:")
subprocess.run(
    ["git", "-C", str(repo_root), "log", "-1", "--oneline"],
    check=True,
)

print("\nPASS: repository cloned.")

Cloning into '/kaggle/working/SpaMGCL'...


Project root:
/kaggle/working/SpaMGCL/SpaMGCL

Commit:
b4abd3d Add files via upload

PASS: repository cloned.


## Cell 4：检查 5 个正式数据集配置和 BSRR 代码

In [4]:
from pathlib import Path
import os
import yaml

project_root = Path("/kaggle/working/SpaMGCL/SpaMGCL")
os.chdir(project_root)

base_configs = {
    "hlna1": "configs/final_clean/hlna1_clean_200.yaml",
    "d1": "configs/final_clean/d1_clean_200.yaml",
    "e185": "configs/final_clean/e185_clean_200.yaml",
    "s2e15": "configs/final_clean/s2e15_clean_200.yaml",
    "s2e18": "configs/final_clean/s2e18_clean_200.yaml",
}

required_code = [
    "src/clustering/refinement.py",
    "src/clustering/predict.py",
    "experiments/run_exp.py",
    "scripts/audit_run.py",
    "tests/test_bsrr.py",
]

print("===== Code =====")

for path in required_code:
    assert Path(path).exists(), f"缺少: {path}"
    print("OK:", path)

print("\n===== Base configs =====")

for key, path in base_configs.items():
    assert Path(path).exists(), f"缺少: {path}"

    with open(path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    print(
        f"{key:6s} | "
        f"dataset={cfg['experiment']['dataset']} | "
        f"clusters={cfg['clustering']['n_clusters']} | "
        f"BSRR={cfg['refinement']['enabled']} | "
        f"k={cfg['refinement']['spatial_k']} | "
        f"n_init={cfg['clustering']['n_init']} | "
        f"kmeans_seed={cfg['clustering']['random_state']}"
    )

    assert cfg["refinement"]["enabled"] is True
    assert cfg["refinement"]["method"] == "bsrr"
    assert int(cfg["refinement"]["spatial_k"]) == 3

    assert int(cfg["clustering"]["n_init"]) == 20
    assert int(cfg["clustering"]["random_state"]) == 0

print("\nPASS: 5 个正式 base configs 正常。")

===== Code =====
OK: src/clustering/refinement.py
OK: src/clustering/predict.py
OK: experiments/run_exp.py
OK: scripts/audit_run.py
OK: tests/test_bsrr.py

===== Base configs =====
hlna1  | dataset=HLN-A1 | clusters=10 | BSRR=True | k=3 | n_init=20 | kmeans_seed=0
d1     | dataset=HLN-D1 | clusters=11 | BSRR=True | k=3 | n_init=20 | kmeans_seed=0
e185   | dataset=E18.5 | clusters=14 | BSRR=True | k=3 | n_init=20 | kmeans_seed=0
s2e15  | dataset=S2-E15 | clusters=15 | BSRR=True | k=3 | n_init=20 | kmeans_seed=0
s2e18  | dataset=S2-E18 | clusters=16 | BSRR=True | k=3 | n_init=20 | kmeans_seed=0

PASS: 5 个正式 base configs 正常。


## Cell 5：检查正式数据集是否已挂载

In [5]:
from pathlib import Path

data_root = Path("/kaggle/input/datasets/wuvdji/smgc-data")

print("Data root:", data_root)
print("Exists:", data_root.exists())

assert data_root.exists(), "没有找到 smgc-data，请先 Add Input。"

print("\nTop-level contents:")

for p in sorted(data_root.iterdir()):
    print(" -", p.name)

print("\nPASS: 数据集已挂载。")

Data root: /kaggle/input/datasets/wuvdji/smgc-data
Exists: True

Top-level contents:
 - E18.5_mouse_brain
 - Human_Lymph_Nodes
 - Mouse_Embryos_S2
 - simulation

PASS: 数据集已挂载。


## Cell 6：生成 5 数据集 × 10 seeds × 50 epochs 正式配置

In [6]:
from pathlib import Path
import copy
import yaml

project_root = Path("/kaggle/working/SpaMGCL/SpaMGCL")

base_configs = {
    "hlna1": "configs/final_clean/hlna1_clean_200.yaml",
    "d1": "configs/final_clean/d1_clean_200.yaml",
    "e185": "configs/final_clean/e185_clean_200.yaml",
    "s2e15": "configs/final_clean/s2e15_clean_200.yaml",
    "s2e18": "configs/final_clean/s2e18_clean_200.yaml",
}

generated_root = project_root / "configs/formal_50ep"
generated_root.mkdir(parents=True, exist_ok=True)

generated_configs = {}

for dataset_key, base_rel in base_configs.items():

    base_path = project_root / base_rel

    with base_path.open("r", encoding="utf-8") as f:
        base_cfg = yaml.safe_load(f)

    generated_configs[dataset_key] = []

    for seed in range(10):

        cfg = copy.deepcopy(base_cfg)

        # 正式训练 seed
        cfg["experiment"]["seed"] = seed

        # 统一 50 epochs
        cfg["experiment"]["epochs"] = 50
        cfg["training"]["epochs"] = 50

        # 每个 seed 独立实验名，避免覆盖
        experiment_name = f"{dataset_key}_formal_50ep_seed{seed}"

        cfg["experiment"]["name"] = experiment_name

        # 再次冻结正式 readout
        cfg["clustering"]["n_init"] = 20
        cfg["clustering"]["random_state"] = 0

        cfg["refinement"]["enabled"] = True
        cfg["refinement"]["method"] = "bsrr"
        cfg["refinement"]["spatial_k"] = 3

        out_path = generated_root / f"{experiment_name}.yaml"

        with out_path.open("w", encoding="utf-8") as f:
            yaml.safe_dump(
                cfg,
                f,
                sort_keys=False,
                allow_unicode=True,
            )

        generated_configs[dataset_key].append(out_path)

print("Generated configs:")

total = 0

for dataset_key, paths in generated_configs.items():
    print(f"{dataset_key:6s}: {len(paths)} configs")
    total += len(paths)

print("\nTotal:", total)

assert total == 50

print("\nPASS: 共生成 50 个正式配置。")

Generated configs:
hlna1 : 10 configs
d1    : 10 configs
e185  : 10 configs
s2e15 : 10 configs
s2e18 : 10 configs

Total: 50

PASS: 共生成 50 个正式配置。


## Cell 7：抽查正式 50-epoch 配置

In [7]:
from pprint import pprint
import yaml

check_paths = [
    "configs/formal_50ep/hlna1_formal_50ep_seed0.yaml",
    "configs/formal_50ep/e185_formal_50ep_seed5.yaml",
    "configs/formal_50ep/s2e18_formal_50ep_seed9.yaml",
]

for path in check_paths:

    print("\n================================")
    print(path)

    with open(path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    print("experiment:")
    pprint(cfg["experiment"])

    print("training epochs:", cfg["training"]["epochs"])

    print("clustering:")
    pprint(cfg["clustering"])

    print("refinement:")
    pprint(cfg["refinement"])

    assert int(cfg["training"]["epochs"]) == 50
    assert int(cfg["experiment"]["epochs"]) == 50

    assert int(cfg["clustering"]["n_init"]) == 20
    assert int(cfg["clustering"]["random_state"]) == 0

    assert cfg["refinement"]["enabled"] is True
    assert cfg["refinement"]["method"] == "bsrr"
    assert int(cfg["refinement"]["spatial_k"]) == 3

print("\nPASS: 正式配置抽查通过。")


configs/formal_50ep/hlna1_formal_50ep_seed0.yaml
experiment:
{'dataset': 'HLN-A1',
 'epochs': 50,
 'name': 'hlna1_formal_50ep_seed0',
 'phase': 'DIAG_REPRESENTATION_CAPACITY',
 'seed': 0,
 'warm_up_epochs': 10}
training epochs: 50
clustering:
{'embedding': 'concat_z',
 'method': 'kmeans',
 'n_clusters': 10,
 'n_init': 20,
 'random_state': 0}
refinement:
{'enabled': True, 'method': 'bsrr', 'spatial_k': 3}

configs/formal_50ep/e185_formal_50ep_seed5.yaml
experiment:
{'dataset': 'E18.5',
 'epochs': 50,
 'name': 'e185_formal_50ep_seed5',
 'phase': 'P1',
 'seed': 5,
 'warm_up_epochs': 10}
training epochs: 50
clustering:
{'embedding': 'concat_z',
 'method': 'kmeans',
 'n_clusters': 14,
 'n_init': 20,
 'random_state': 0}
refinement:
{'enabled': True, 'method': 'bsrr', 'spatial_k': 3}

configs/formal_50ep/s2e18_formal_50ep_seed9.yaml
experiment:
{'dataset': 'S2-E18',
 'epochs': 50,
 'name': 's2e18_formal_50ep_seed9',
 'phase': 'P1',
 'seed': 9,
 'warm_up_epochs': 10}
training epochs: 50
clust

## Cell 8：定义正式实验 run + audit 函数

In [8]:
import os
import sys
import yaml
import subprocess
from pathlib import Path

project_root = Path("/kaggle/working/SpaMGCL/SpaMGCL")
os.chdir(project_root)

def run_and_audit(config_path):
    config_path = Path(config_path)

    with config_path.open("r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    experiment_name = cfg["experiment"]["name"]
    seed = int(cfg["experiment"]["seed"])
    dataset = cfg["experiment"]["dataset"]

    output_dir = project_root / "results_clean" / experiment_name
    metrics_path = output_dir / "metrics.json"

    print("\n" + "=" * 80)
    print(f"Dataset : {dataset}")
    print(f"Seed    : {seed}")
    print(f"Config  : {config_path}")
    print(f"Output  : {output_dir}")
    print("=" * 80)

    # 允许 Kaggle 中断后续跑：
    # 已经完整完成的 run 不重新训练，但仍重新 audit
    if metrics_path.exists():
        print("已有 metrics.json，跳过训练，直接 audit。")
    else:
        subprocess.run(
            [
                sys.executable,
                "experiments/run_exp.py",
                "--config",
                str(config_path),
            ],
            check=True,
        )

    print("\nRunning audit...")

    subprocess.run(
        [
            sys.executable,
            "scripts/audit_run.py",
            str(output_dir),
        ],
        check=True,
    )

    print(f"\nPASS: {experiment_name}")

    return output_dir


print("PASS: formal runner ready.")

PASS: formal runner ready.


## Cell 9：正式运行 HLN-A1 × 10 training seeds

In [9]:
from pathlib import Path

config_root = Path("configs/formal_50ep")

hlna1_outputs = []

for seed in range(10):

    config_path = (
        config_root
        / f"hlna1_formal_50ep_seed{seed}.yaml"
    )

    assert config_path.exists(), f"缺少配置: {config_path}"

    output_dir = run_and_audit(config_path)

    hlna1_outputs.append(output_dir)

print("\n" + "=" * 80)
print("HLN-A1 ALL 10 SEEDS COMPLETED")
print("=" * 80)

for p in hlna1_outputs:
    print(p.name)


Dataset : HLN-A1
Seed    : 0
Config  : configs/formal_50ep/hlna1_formal_50ep_seed0.yaml
Output  : /kaggle/working/SpaMGCL/SpaMGCL/results_clean/hlna1_formal_50ep_seed0


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: HLN-A1 | spots=3484 | device=cuda
Modalities: RNA, ADT | label=Spatial_Label
Spatial graph: shape=(3484, 3484), nnz=13206
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/050 | total=8.411676 | rec=0.252383 | mgcl=8.159293 | cluster=2.951158 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=4.887e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.950 | gradC=0.000e+00
epoch 002/050 | total=8.400129 | rec=0.242314 | mgcl=8.157816 | cluster=2.950293 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=2.620e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.956 | gradC=0.000e+00
epoch 003/050 | total=8.389498 | rec=0.232629 | mgcl=8.156869 | cluster=2.949508 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=4.571e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.962 | gradC=0.000e+00
epoch 00

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: HLN-A1 | spots=3484 | device=cuda
Modalities: RNA, ADT | label=Spatial_Label
Spatial graph: shape=(3484, 3484), nnz=13206
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/050 | total=8.396720 | rec=0.237463 | mgcl=8.159257 | cluster=2.950006 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=2.644e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.959 | gradC=0.000e+00
epoch 002/050 | total=8.386283 | rec=0.228770 | mgcl=8.157513 | cluster=2.949127 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=2.560e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.966 | gradC=0.000e+00
epoch 003/050 | total=8.376886 | rec=0.220448 | mgcl=8.156439 | cluster=2.948355 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=2.735e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.972 | gradC=0.000e+00
epoch 00

...

Dataset: HLN-A1 | spots=3484 | device=cuda
Modalities: RNA, ADT | label=Spatial_Label
Spatial graph: shape=(3484, 3484), nnz=13206
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/050 | total=8.447705 | rec=0.289945 | mgcl=8.157760 | cluster=2.951674 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=3.613e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.950 | gradC=0.000e+00
epoch 002/050 | total=8.435236 | rec=0.278373 | mgcl=8.156863 | cluster=2.950777 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=4.174e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.957 | gradC=0.000e+00
epoch 003/050 | total=8.423659 | rec=0.267463 | mgcl=8.156197 | cluster=2.949970 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=5.050e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.962 | gradC=0.000e+00
epoch 00

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: HLN-A1 | spots=3484 | device=cuda
Modalities: RNA, ADT | label=Spatial_Label
Spatial graph: shape=(3484, 3484), nnz=13206
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/050 | total=8.417413 | rec=0.259766 | mgcl=8.157647 | cluster=2.950842 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=6.794e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.953 | gradC=0.000e+00
epoch 002/050 | total=8.405575 | rec=0.249104 | mgcl=8.156470 | cluster=2.949901 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=9.827e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.961 | gradC=0.000e+00
epoch 003/050 | total=8.394566 | rec=0.239003 | mgcl=8.155562 | cluster=2.949052 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.031e-02 | neg_count=12134772 | snf_masked_positions=0 | effC=9.967 | gradC=0.000e+00
epoch 00

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: HLN-A1 | spots=3484 | device=cuda
Modalities: RNA, ADT | label=Spatial_Label
Spatial graph: shape=(3484, 3484), nnz=13206
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/050 | total=8.392878 | rec=0.234356 | mgcl=8.158521 | cluster=2.951650 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=3.590e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.936 | gradC=0.000e+00
epoch 002/050 | total=8.382640 | rec=0.225641 | mgcl=8.157000 | cluster=2.950634 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=6.190e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.945 | gradC=0.000e+00
epoch 003/050 | total=8.373120 | rec=0.217257 | mgcl=8.155863 | cluster=2.949736 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=7.015e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.953 | gradC=0.000e+00
epoch 00

## Cell 10：汇总 HLN-A1 10-seed 正式结果

In [10]:
import json
import numpy as np
from pathlib import Path

result_root = Path(
    "/kaggle/working/SpaMGCL/SpaMGCL/results_clean"
)

records = []

for seed in range(10):
    run_dir = result_root / f"hlna1_formal_50ep_seed{seed}"
    metrics_path = run_dir / "metrics.json"

    assert metrics_path.exists(), f"缺少: {metrics_path}"

    with metrics_path.open("r", encoding="utf-8") as f:
        metrics = json.load(f)

    ari = float(metrics["ARI"])
    nmi = float(metrics["NMI"])

    records.append((seed, ari, nmi))

print("===== HLN-A1 per-seed results =====")

for seed, ari, nmi in records:
    print(
        f"seed {seed}: "
        f"ARI={ari:.6f} | "
        f"NMI={nmi:.6f}"
    )

aris = np.array([r[1] for r in records])
nmis = np.array([r[2] for r in records])

print("\n===== HLN-A1 10-run summary =====")

print(
    f"ARI = {aris.mean():.6f} ± {aris.std(ddof=0):.6f}"
)
print(
    f"NMI = {nmis.mean():.6f} ± {nmis.std(ddof=0):.6f}"
)

print("\nRange:")
print(
    f"ARI min={aris.min():.6f}, max={aris.max():.6f}"
)
print(
    f"NMI min={nmis.min():.6f}, max={nmis.max():.6f}"
)

assert len(records) == 10
assert np.isfinite(aris).all()
assert np.isfinite(nmis).all()

print("\nPASS: HLN-A1 10-seed summary complete.")

===== HLN-A1 per-seed results =====
seed 0: ARI=0.239996 | NMI=0.336198
seed 1: ARI=0.264916 | NMI=0.364676
seed 2: ARI=0.209397 | NMI=0.337053
seed 3: ARI=0.227045 | NMI=0.334193
seed 4: ARI=0.231193 | NMI=0.315773
seed 5: ARI=0.235218 | NMI=0.347538
seed 6: ARI=0.260178 | NMI=0.338467
seed 7: ARI=0.238806 | NMI=0.329467
seed 8: ARI=0.229868 | NMI=0.323038
seed 9: ARI=0.262099 | NMI=0.310668

===== HLN-A1 10-run summary =====
ARI = 0.239872 ± 0.016808
NMI = 0.333707 ± 0.014747

Range:
ARI min=0.209397, max=0.264916
NMI min=0.310668, max=0.364676

PASS: HLN-A1 10-seed summary complete.


## Cell 11：统计 HLN-A1 上 BSRR 的 10-seed 平均增益

In [11]:
import json
import numpy as np

from pathlib import Path
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)

result_root = Path(
    "/kaggle/working/SpaMGCL/SpaMGCL/results_clean"
)

records = []

for seed in range(10):

    run_dir = result_root / f"hlna1_formal_50ep_seed{seed}"

    gt = np.load(run_dir / "gt_labels.npy")
    pred_raw = np.load(run_dir / "pred_concat_z_kmeans.npy")
    pred_bsrr = np.load(run_dir / "pred_labels.npy")

    raw_ari = adjusted_rand_score(gt, pred_raw)
    raw_nmi = normalized_mutual_info_score(
        gt,
        pred_raw,
        average_method="max",
    )

    bsrr_ari = adjusted_rand_score(gt, pred_bsrr)
    bsrr_nmi = normalized_mutual_info_score(
        gt,
        pred_bsrr,
        average_method="max",
    )

    records.append(
        {
            "seed": seed,
            "raw_ari": raw_ari,
            "bsrr_ari": bsrr_ari,
            "delta_ari": bsrr_ari - raw_ari,
            "raw_nmi": raw_nmi,
            "bsrr_nmi": bsrr_nmi,
            "delta_nmi": bsrr_nmi - raw_nmi,
        }
    )

print("===== Per-seed BSRR effect =====")

for r in records:
    print(
        f"seed {r['seed']}: "
        f"ARI {r['raw_ari']:.6f} -> {r['bsrr_ari']:.6f} "
        f"({r['delta_ari']:+.6f}) | "
        f"NMI {r['raw_nmi']:.6f} -> {r['bsrr_nmi']:.6f} "
        f"({r['delta_nmi']:+.6f})"
    )

raw_ari = np.array([r["raw_ari"] for r in records])
bsrr_ari = np.array([r["bsrr_ari"] for r in records])
delta_ari = np.array([r["delta_ari"] for r in records])

raw_nmi = np.array([r["raw_nmi"] for r in records])
bsrr_nmi = np.array([r["bsrr_nmi"] for r in records])
delta_nmi = np.array([r["delta_nmi"] for r in records])

print("\n===== 10-seed mean =====")

print(
    f"Raw ARI  = {raw_ari.mean():.6f} ± {raw_ari.std(ddof=0):.6f}"
)
print(
    f"BSRR ARI = {bsrr_ari.mean():.6f} ± {bsrr_ari.std(ddof=0):.6f}"
)
print(
    f"Delta ARI mean = {delta_ari.mean():+.6f}"
)

print()

print(
    f"Raw NMI  = {raw_nmi.mean():.6f} ± {raw_nmi.std(ddof=0):.6f}"
)
print(
    f"BSRR NMI = {bsrr_nmi.mean():.6f} ± {bsrr_nmi.std(ddof=0):.6f}"
)
print(
    f"Delta NMI mean = {delta_nmi.mean():+.6f}"
)

print("\n===== Positive seeds =====")
print(
    "ARI improved:",
    int((delta_ari > 0).sum()),
    "/ 10",
)
print(
    "NMI improved:",
    int((delta_nmi > 0).sum()),
    "/ 10",
)

print("\nPASS: HLN-A1 BSRR effect summary complete.")

===== Per-seed BSRR effect =====
seed 0: ARI 0.241407 -> 0.239996 (-0.001411) | NMI 0.336315 -> 0.336198 (-0.000117)
seed 1: ARI 0.260963 -> 0.264916 (+0.003953) | NMI 0.358484 -> 0.364676 (+0.006192)
seed 2: ARI 0.217936 -> 0.209397 (-0.008539) | NMI 0.344763 -> 0.337053 (-0.007710)
seed 3: ARI 0.226175 -> 0.227045 (+0.000870) | NMI 0.330264 -> 0.334193 (+0.003929)
seed 4: ARI 0.239151 -> 0.231193 (-0.007959) | NMI 0.319991 -> 0.315773 (-0.004218)
seed 5: ARI 0.233845 -> 0.235218 (+0.001373) | NMI 0.342769 -> 0.347538 (+0.004769)
seed 6: ARI 0.264461 -> 0.260178 (-0.004283) | NMI 0.329640 -> 0.338467 (+0.008827)
seed 7: ARI 0.243237 -> 0.238806 (-0.004431) | NMI 0.326476 -> 0.329467 (+0.002991)
seed 8: ARI 0.221050 -> 0.229868 (+0.008818) | NMI 0.315681 -> 0.323038 (+0.007357)
seed 9: ARI 0.253635 -> 0.262099 (+0.008464) | NMI 0.305180 -> 0.310668 (+0.005488)

===== 10-seed mean =====
Raw ARI  = 0.240186 ± 0.015204
BSRR ARI = 0.239872 ± 0.016808
Delta ARI mean = -0.000315

Raw NMI  = 

## Cell 12：正式运行 HLN-D1 × 10 training seeds

In [12]:
from pathlib import Path

config_root = Path("configs/formal_50ep")

d1_outputs = []

for seed in range(10):

    config_path = (
        config_root
        / f"d1_formal_50ep_seed{seed}.yaml"
    )

    assert config_path.exists(), f"缺少配置: {config_path}"

    output_dir = run_and_audit(config_path)

    d1_outputs.append(output_dir)

print("\n" + "=" * 80)
print("HLN-D1 ALL 10 SEEDS COMPLETED")
print("=" * 80)

for p in d1_outputs:
    print(p.name)


Dataset : HLN-D1
Seed    : 0
Config  : configs/formal_50ep/d1_formal_50ep_seed0.yaml
Output  : /kaggle/working/SpaMGCL/SpaMGCL/results_clean/d1_formal_50ep_seed0


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: HLN-D1 | spots=3359 | device=cuda
Modalities: RNA, ADT | label=Spatial_Label
Spatial graph: shape=(3359, 3359), nnz=12744
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/050 | total=8.370173 | rec=0.248054 | mgcl=8.122119 | cluster=3.050508 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=4.870e-03 | neg_count=11279522 | snf_masked_positions=0 | effC=10.952 | gradC=0.000e+00
epoch 002/050 | total=8.359020 | rec=0.238144 | mgcl=8.120875 | cluster=3.049739 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=2.930e-03 | neg_count=11279522 | snf_masked_positions=0 | effC=10.958 | gradC=0.000e+00
epoch 003/050 | total=8.348646 | rec=0.228612 | mgcl=8.120034 | cluster=3.049047 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=2.583e-03 | neg_count=11279522 | snf_masked_positions=0 | effC=10.963 | gradC=0.000e+00
epoch

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: HLN-D1 | spots=3359 | device=cuda
Modalities: RNA, ADT | label=Spatial_Label
Spatial graph: shape=(3359, 3359), nnz=12744
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/050 | total=8.349872 | rec=0.226992 | mgcl=8.122880 | cluster=3.048808 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=2.623e-03 | neg_count=11279522 | snf_masked_positions=0 | effC=10.970 | gradC=0.000e+00
epoch 002/050 | total=8.339814 | rec=0.218547 | mgcl=8.121267 | cluster=3.048234 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=2.568e-03 | neg_count=11279522 | snf_masked_positions=0 | effC=10.975 | gradC=0.000e+00
epoch 003/050 | total=8.330775 | rec=0.210469 | mgcl=8.120307 | cluster=3.047729 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=2.795e-03 | neg_count=11279522 | snf_masked_positions=0 | effC=10.978 | gradC=0.000e+00
epoch

...

Dataset: HLN-D1 | spots=3359 | device=cuda
Modalities: RNA, ADT | label=Spatial_Label
Spatial graph: shape=(3359, 3359), nnz=12744
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/050 | total=8.400757 | rec=0.279445 | mgcl=8.121311 | cluster=3.051596 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=3.611e-03 | neg_count=11279522 | snf_masked_positions=0 | effC=10.946 | gradC=0.000e+00
epoch 002/050 | total=8.388640 | rec=0.268163 | mgcl=8.120478 | cluster=3.050722 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=2.620e-03 | neg_count=11279522 | snf_masked_positions=0 | effC=10.953 | gradC=0.000e+00
epoch 003/050 | total=8.377420 | rec=0.257548 | mgcl=8.119873 | cluster=3.049953 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.990e-03 | neg_count=11279522 | snf_masked_positions=0 | effC=10.959 | gradC=0.000e+00
epoch

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: HLN-D1 | spots=3359 | device=cuda
Modalities: RNA, ADT | label=Spatial_Label
Spatial graph: shape=(3359, 3359), nnz=12744
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/050 | total=8.380973 | rec=0.259627 | mgcl=8.121346 | cluster=3.053863 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=6.816e-03 | neg_count=11279522 | snf_masked_positions=0 | effC=10.916 | gradC=0.000e+00
epoch 002/050 | total=8.369314 | rec=0.249122 | mgcl=8.120192 | cluster=3.052666 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.013e-02 | neg_count=11279522 | snf_masked_positions=0 | effC=10.926 | gradC=0.000e+00
epoch 003/050 | total=8.358519 | rec=0.239153 | mgcl=8.119365 | cluster=3.051585 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.161e-02 | neg_count=11279522 | snf_masked_positions=0 | effC=10.936 | gradC=0.000e+00
epoch

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: HLN-D1 | spots=3359 | device=cuda
Modalities: RNA, ADT | label=Spatial_Label
Spatial graph: shape=(3359, 3359), nnz=12744
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/050 | total=8.350972 | rec=0.229622 | mgcl=8.121350 | cluster=3.049772 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=3.657e-03 | neg_count=11279522 | snf_masked_positions=0 | effC=10.950 | gradC=0.000e+00
epoch 002/050 | total=8.340958 | rec=0.221065 | mgcl=8.119893 | cluster=3.049099 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=5.784e-03 | neg_count=11279522 | snf_masked_positions=0 | effC=10.956 | gradC=0.000e+00
epoch 003/050 | total=8.331631 | rec=0.212854 | mgcl=8.118776 | cluster=3.048502 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=6.388e-03 | neg_count=11279522 | snf_masked_positions=0 | effC=10.962 | gradC=0.000e+00
epoch

## Cell 13：定义正式结果汇总函数并汇总 HLN-D1

In [13]:
import json
import numpy as np

from pathlib import Path
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)

result_root = Path(
    "/kaggle/working/SpaMGCL/SpaMGCL/results_clean"
)


def summarize_dataset(dataset_key):

    records = []

    for seed in range(10):

        run_dir = (
            result_root
            / f"{dataset_key}_formal_50ep_seed{seed}"
        )

        metrics_path = run_dir / "metrics.json"

        assert metrics_path.exists(), f"缺少: {metrics_path}"

        with metrics_path.open("r", encoding="utf-8") as f:
            metrics = json.load(f)

        gt = np.load(run_dir / "gt_labels.npy")
        pred_raw = np.load(
            run_dir / "pred_concat_z_kmeans.npy"
        )
        pred_bsrr = np.load(
            run_dir / "pred_labels.npy"
        )

        raw_ari = adjusted_rand_score(
            gt,
            pred_raw,
        )

        raw_nmi = normalized_mutual_info_score(
            gt,
            pred_raw,
            average_method="max",
        )

        bsrr_ari = adjusted_rand_score(
            gt,
            pred_bsrr,
        )

        bsrr_nmi = normalized_mutual_info_score(
            gt,
            pred_bsrr,
            average_method="max",
        )

        # metrics.json 必须和正式 prediction 一致
        assert np.isclose(
            bsrr_ari,
            float(metrics["ARI"]),
        )

        assert np.isclose(
            bsrr_nmi,
            float(metrics["NMI"]),
        )

        records.append(
            {
                "seed": seed,
                "raw_ari": raw_ari,
                "raw_nmi": raw_nmi,
                "bsrr_ari": bsrr_ari,
                "bsrr_nmi": bsrr_nmi,
                "delta_ari": bsrr_ari - raw_ari,
                "delta_nmi": bsrr_nmi - raw_nmi,
            }
        )

    print(
        f"===== {dataset_key.upper()} per-seed results ====="
    )

    for r in records:
        print(
            f"seed {r['seed']}: "
            f"ARI={r['bsrr_ari']:.6f} | "
            f"NMI={r['bsrr_nmi']:.6f} | "
            f"ΔARI={r['delta_ari']:+.6f} | "
            f"ΔNMI={r['delta_nmi']:+.6f}"
        )

    raw_ari = np.array(
        [r["raw_ari"] for r in records]
    )

    raw_nmi = np.array(
        [r["raw_nmi"] for r in records]
    )

    bsrr_ari = np.array(
        [r["bsrr_ari"] for r in records]
    )

    bsrr_nmi = np.array(
        [r["bsrr_nmi"] for r in records]
    )

    delta_ari = np.array(
        [r["delta_ari"] for r in records]
    )

    delta_nmi = np.array(
        [r["delta_nmi"] for r in records]
    )

    print(
        f"\n===== {dataset_key.upper()} 10-run summary ====="
    )

    print(
        f"Official ARI = "
        f"{bsrr_ari.mean():.6f} ± "
        f"{bsrr_ari.std(ddof=0):.6f}"
    )

    print(
        f"Official NMI = "
        f"{bsrr_nmi.mean():.6f} ± "
        f"{bsrr_nmi.std(ddof=0):.6f}"
    )

    print("\n===== Raw concat-Z =====")

    print(
        f"Raw ARI = "
        f"{raw_ari.mean():.6f} ± "
        f"{raw_ari.std(ddof=0):.6f}"
    )

    print(
        f"Raw NMI = "
        f"{raw_nmi.mean():.6f} ± "
        f"{raw_nmi.std(ddof=0):.6f}"
    )

    print("\n===== BSRR effect =====")

    print(
        f"Mean ΔARI = "
        f"{delta_ari.mean():+.6f}"
    )

    print(
        f"Mean ΔNMI = "
        f"{delta_nmi.mean():+.6f}"
    )

    print(
        "ARI improved:",
        int((delta_ari > 0).sum()),
        "/ 10",
    )

    print(
        "NMI improved:",
        int((delta_nmi > 0).sum()),
        "/ 10",
    )

    summary = {
        "dataset": dataset_key,
        "ari_mean": float(bsrr_ari.mean()),
        "ari_std": float(bsrr_ari.std(ddof=0)),
        "nmi_mean": float(bsrr_nmi.mean()),
        "nmi_std": float(bsrr_nmi.std(ddof=0)),
        "raw_ari_mean": float(raw_ari.mean()),
        "raw_nmi_mean": float(raw_nmi.mean()),
        "delta_ari_mean": float(delta_ari.mean()),
        "delta_nmi_mean": float(delta_nmi.mean()),
        "ari_positive_seeds": int(
            (delta_ari > 0).sum()
        ),
        "nmi_positive_seeds": int(
            (delta_nmi > 0).sum()
        ),
    }

    return summary


d1_summary = summarize_dataset("d1")

print("\nPASS: HLN-D1 summary complete.")

===== D1 per-seed results =====
seed 0: ARI=0.188883 | NMI=0.253237 | ΔARI=+0.009491 | ΔNMI=+0.003043
seed 1: ARI=0.171797 | NMI=0.249671 | ΔARI=-0.000660 | ΔNMI=+0.001837
seed 2: ARI=0.261958 | NMI=0.343008 | ΔARI=+0.040638 | ΔNMI=+0.028858
seed 3: ARI=0.209625 | NMI=0.308442 | ΔARI=+0.013904 | ΔNMI=+0.005268
seed 4: ARI=0.174790 | NMI=0.240849 | ΔARI=+0.011113 | ΔNMI=+0.007207
seed 5: ARI=0.179664 | NMI=0.266059 | ΔARI=+0.007559 | ΔNMI=+0.016965
seed 6: ARI=0.219505 | NMI=0.282286 | ΔARI=+0.003474 | ΔNMI=+0.005090
seed 7: ARI=0.290896 | NMI=0.308691 | ΔARI=+0.038864 | ΔNMI=+0.026016
seed 8: ARI=0.212463 | NMI=0.265247 | ΔARI=-0.003247 | ΔNMI=+0.000348
seed 9: ARI=0.210420 | NMI=0.301660 | ΔARI=+0.000906 | ΔNMI=+0.002631

===== D1 10-run summary =====
Official ARI = 0.212000 ± 0.036560
Official NMI = 0.281915 ± 0.031017

===== Raw concat-Z =====
Raw ARI = 0.199796 ± 0.026561
Raw NMI = 0.272189 ± 0.025934

===== BSRR effect =====
Mean ΔARI = +0.012204
Mean ΔNMI = +0.009726
ARI improved

## Cell 14：正式运行 E18.5 × 10 training seeds

In [14]:
from pathlib import Path

config_root = Path("configs/formal_50ep")

e185_outputs = []

for seed in range(10):

    config_path = (
        config_root
        / f"e185_formal_50ep_seed{seed}.yaml"
    )

    assert config_path.exists(), f"缺少配置: {config_path}"

    output_dir = run_and_audit(config_path)

    e185_outputs.append(output_dir)

print("\n" + "=" * 80)
print("E18.5 ALL 10 SEEDS COMPLETED")
print("=" * 80)

for p in e185_outputs:
    print(p.name)


Dataset : E18.5
Seed    : 0
Config  : configs/formal_50ep/e185_formal_50ep_seed0.yaml
Output  : /kaggle/working/SpaMGCL/SpaMGCL/results_clean/e185_formal_50ep_seed0
Dataset: E18.5 | spots=2129 | device=cuda
Modalities: RNA, ATAC | label=Combined_Clusters
Spatial graph: shape=(2129, 2129), nnz=7784
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/050 | total=23.166977 | rec=0.160814 | mgcl=7.668721 | cluster=3.298035 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=8.967e-03 | neg_count=4530512 | snf_masked_positions=0 | effC=13.976 | gradC=0.000e+00
epoch 002/050 | total=23.126158 | rec=0.148333 | mgcl=7.659275 | cluster=3.297572 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=7.602e-03 | neg_count=4530512 | snf_masked_positions=0 | effC=13.982 | gradC=0.000e+00
epoch 003/050 | total=23.084679 | rec=0.137188 | mgcl=7.649164 | cluster=3.297174 | spatial_loss=0.000000

## Cell 15：汇总 E18.5 10-seed 正式结果与 BSRR 增益

In [15]:
e185_summary = summarize_dataset("e185")

print("\nPASS: E18.5 summary complete.")

===== E185 per-seed results =====
seed 0: ARI=0.443555 | NMI=0.525320 | ΔARI=+0.003999 | ΔNMI=+0.000594
seed 1: ARI=0.345591 | NMI=0.493158 | ΔARI=+0.011411 | ΔNMI=+0.008422
seed 2: ARI=0.384299 | NMI=0.510932 | ΔARI=+0.017902 | ΔNMI=+0.004900
seed 3: ARI=0.338779 | NMI=0.549492 | ΔARI=+0.004175 | ΔNMI=+0.033219
seed 4: ARI=0.435727 | NMI=0.561754 | ΔARI=+0.068152 | ΔNMI=+0.023691
seed 5: ARI=0.361497 | NMI=0.523603 | ΔARI=-0.021763 | ΔNMI=+0.003394
seed 6: ARI=0.441354 | NMI=0.536783 | ΔARI=+0.022353 | ΔNMI=+0.005053
seed 7: ARI=0.445289 | NMI=0.530690 | ΔARI=+0.025862 | ΔNMI=+0.008443
seed 8: ARI=0.361421 | NMI=0.528994 | ΔARI=+0.006922 | ΔNMI=+0.004796
seed 9: ARI=0.374214 | NMI=0.501891 | ΔARI=+0.021925 | ΔNMI=-0.008858

===== E185 10-run summary =====
Official ARI = 0.393173 ± 0.041303
Official NMI = 0.526262 ± 0.019672

===== Raw concat-Z =====
Raw ARI = 0.377079 ± 0.035307
Raw NMI = 0.517896 ± 0.014152

===== BSRR effect =====
Mean ΔARI = +0.016094
Mean ΔNMI = +0.008365
ARI impr

## Cell 16：正式运行 S2-E15 × 10 training seeds

In [16]:
from pathlib import Path

config_root = Path("configs/formal_50ep")

s2e15_outputs = []

for seed in range(10):

    config_path = (
        config_root
        / f"s2e15_formal_50ep_seed{seed}.yaml"
    )

    assert config_path.exists(), f"缺少配置: {config_path}"

    output_dir = run_and_audit(config_path)

    s2e15_outputs.append(output_dir)

print("\n" + "=" * 80)
print("S2-E15 ALL 10 SEEDS COMPLETED")
print("=" * 80)

for p in s2e15_outputs:
    print(p.name)


Dataset : S2-E15
Seed    : 0
Config  : configs/formal_50ep/s2e15_formal_50ep_seed0.yaml
Output  : /kaggle/working/SpaMGCL/SpaMGCL/results_clean/s2e15_formal_50ep_seed0


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: S2-E15 | spots=1939 | device=cuda
Modalities: RNA, ATAC | label=Spatial_Label
Spatial graph: shape=(1939, 1939), nnz=7062
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/050 | total=22.891996 | rec=0.165808 | mgcl=7.575396 | cluster=3.369660 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=8.919e-03 | neg_count=3757782 | snf_masked_positions=0 | effC=14.972 | gradC=0.000e+00
epoch 002/050 | total=22.847919 | rec=0.153040 | mgcl=7.564960 | cluster=3.369064 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=7.980e-03 | neg_count=3757782 | snf_masked_positions=0 | effC=14.979 | gradC=0.000e+00
epoch 003/050 | total=22.801908 | rec=0.141745 | mgcl=7.553388 | cluster=3.368593 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=7.761e-03 | neg_count=3757782 | snf_masked_positions=0 | effC=14.985 | gradC=0.000e+00
epoch

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: S2-E15 | spots=1939 | device=cuda
Modalities: RNA, ATAC | label=Spatial_Label
Spatial graph: shape=(1939, 1939), nnz=7062
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/050 | total=22.889145 | rec=0.161898 | mgcl=7.575749 | cluster=3.369750 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=5.955e-03 | neg_count=3757782 | snf_masked_positions=0 | effC=14.974 | gradC=0.000e+00
epoch 002/050 | total=22.842108 | rec=0.149705 | mgcl=7.564134 | cluster=3.369132 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=4.816e-03 | neg_count=3757782 | snf_masked_positions=0 | effC=14.980 | gradC=0.000e+00
epoch 003/050 | total=22.789713 | rec=0.138926 | mgcl=7.550262 | cluster=3.368660 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=6.825e-03 | neg_count=3757782 | snf_masked_positions=0 | effC=14.985 | gradC=0.000e+00
epoch

...

Dataset: S2-E15 | spots=1939 | device=cuda
Modalities: RNA, ATAC | label=Spatial_Label
Spatial graph: shape=(1939, 1939), nnz=7062
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/050 | total=22.894012 | rec=0.162945 | mgcl=7.577023 | cluster=3.368985 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.110e-02 | neg_count=3757782 | snf_masked_positions=0 | effC=14.979 | gradC=0.000e+00
epoch 002/050 | total=22.857248 | rec=0.150657 | mgcl=7.568863 | cluster=3.368510 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.404e-02 | neg_count=3757782 | snf_masked_positions=0 | effC=14.985 | gradC=0.000e+00
epoch 003/050 | total=22.830038 | rec=0.139443 | mgcl=7.563532 | cluster=3.368177 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.405e-02 | neg_count=3757782 | snf_masked_positions=0 | effC=14.990 | gradC=0.000e+00
epoch

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: S2-E15 | spots=1939 | device=cuda
Modalities: RNA, ATAC | label=Spatial_Label
Spatial graph: shape=(1939, 1939), nnz=7062
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/050 | total=22.896702 | rec=0.163825 | mgcl=7.577625 | cluster=3.369317 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.180e-02 | neg_count=3757782 | snf_masked_positions=0 | effC=14.976 | gradC=0.000e+00
epoch 002/050 | total=22.856598 | rec=0.150955 | mgcl=7.568548 | cluster=3.368789 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.402e-02 | neg_count=3757782 | snf_masked_positions=0 | effC=14.983 | gradC=0.000e+00
epoch 003/050 | total=22.829460 | rec=0.139290 | mgcl=7.563390 | cluster=3.368339 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.329e-02 | neg_count=3757782 | snf_masked_positions=0 | effC=14.988 | gradC=0.000e+00
epoch

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: S2-E15 | spots=1939 | device=cuda
Modalities: RNA, ATAC | label=Spatial_Label
Spatial graph: shape=(1939, 1939), nnz=7062
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/050 | total=22.890804 | rec=0.164342 | mgcl=7.575487 | cluster=3.368723 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=8.708e-03 | neg_count=3757782 | snf_masked_positions=0 | effC=14.983 | gradC=0.000e+00
epoch 002/050 | total=22.857164 | rec=0.151799 | mgcl=7.568455 | cluster=3.368379 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=7.938e-03 | neg_count=3757782 | snf_masked_positions=0 | effC=14.987 | gradC=0.000e+00
epoch 003/050 | total=22.831724 | rec=0.140405 | mgcl=7.563773 | cluster=3.368101 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=8.173e-03 | neg_count=3757782 | snf_masked_positions=0 | effC=14.990 | gradC=0.000e+00
epoch

## Cell 17：汇总 S2-E15 10-seed 正式结果与 BSRR 增益

In [17]:
s2e15_summary = summarize_dataset("s2e15")

print("\nPASS: S2-E15 summary complete.")

===== S2E15 per-seed results =====
seed 0: ARI=0.336883 | NMI=0.531759 | ΔARI=-0.005268 | ΔNMI=+0.016221
seed 1: ARI=0.368609 | NMI=0.544335 | ΔARI=+0.022298 | ΔNMI=+0.005240
seed 2: ARI=0.422500 | NMI=0.564555 | ΔARI=+0.013134 | ΔNMI=+0.006214
seed 3: ARI=0.327815 | NMI=0.514313 | ΔARI=+0.014704 | ΔNMI=+0.012485
seed 4: ARI=0.352979 | NMI=0.556246 | ΔARI=+0.047860 | ΔNMI=+0.030770
seed 5: ARI=0.353717 | NMI=0.545889 | ΔARI=-0.006056 | ΔNMI=+0.004949
seed 6: ARI=0.407569 | NMI=0.570231 | ΔARI=+0.010051 | ΔNMI=+0.025815
seed 7: ARI=0.381713 | NMI=0.545225 | ΔARI=+0.013828 | ΔNMI=+0.016243
seed 8: ARI=0.419142 | NMI=0.564201 | ΔARI=+0.012305 | ΔNMI=+0.012962
seed 9: ARI=0.388275 | NMI=0.537306 | ΔARI=+0.013000 | ΔNMI=+0.011151

===== S2E15 10-run summary =====
Official ARI = 0.375920 ± 0.031838
Official NMI = 0.547406 ± 0.016229

===== Raw concat-Z =====
Raw ARI = 0.362335 ± 0.034635
Raw NMI = 0.533201 ± 0.016128

===== BSRR effect =====
Mean ΔARI = +0.013586
Mean ΔNMI = +0.014205
ARI im

## Cell 18：正式运行 S2-E18 × 10 training seeds

In [18]:
from pathlib import Path

config_root = Path("configs/formal_50ep")

s2e18_outputs = []

for seed in range(10):

    config_path = (
        config_root
        / f"s2e18_formal_50ep_seed{seed}.yaml"
    )

    assert config_path.exists(), f"缺少配置: {config_path}"

    output_dir = run_and_audit(config_path)

    s2e18_outputs.append(output_dir)

print("\n" + "=" * 80)
print("S2-E18 ALL 10 SEEDS COMPLETED")
print("=" * 80)

for p in s2e18_outputs:
    print(p.name)


Dataset : S2-E18
Seed    : 0
Config  : configs/formal_50ep/s2e18_formal_50ep_seed0.yaml
Output  : /kaggle/working/SpaMGCL/SpaMGCL/results_clean/s2e18_formal_50ep_seed0


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: S2-E18 | spots=2248 | device=cuda
Modalities: RNA, ATAC | label=Spatial_Label
Spatial graph: shape=(2248, 2248), nnz=8286
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/050 | total=23.328171 | rec=0.160767 | mgcl=7.722468 | cluster=3.435536 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=8.855e-03 | neg_count=5051256 | snf_masked_positions=0 | effC=15.982 | gradC=0.000e+00
epoch 002/050 | total=23.283112 | rec=0.148299 | mgcl=7.711605 | cluster=3.435190 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=8.357e-03 | neg_count=5051256 | snf_masked_positions=0 | effC=15.987 | gradC=0.000e+00
epoch 003/050 | total=23.231668 | rec=0.137329 | mgcl=7.698113 | cluster=3.434907 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=8.303e-03 | neg_count=5051256 | snf_masked_positions=0 | effC=15.990 | gradC=0.000e+00
epoch

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: S2-E18 | spots=2248 | device=cuda
Modalities: RNA, ATAC | label=Spatial_Label
Spatial graph: shape=(2248, 2248), nnz=8286
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/050 | total=23.324621 | rec=0.155365 | mgcl=7.723085 | cluster=3.436096 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=5.948e-03 | neg_count=5051256 | snf_masked_positions=0 | effC=15.978 | gradC=0.000e+00
epoch 002/050 | total=23.275261 | rec=0.143727 | mgcl=7.710512 | cluster=3.435551 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=4.556e-03 | neg_count=5051256 | snf_masked_positions=0 | effC=15.983 | gradC=0.000e+00
epoch 003/050 | total=23.212103 | rec=0.133637 | mgcl=7.692822 | cluster=3.435139 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=4.940e-03 | neg_count=5051256 | snf_masked_positions=0 | effC=15.988 | gradC=0.000e+00
epoch

...

Dataset: S2-E18 | spots=2248 | device=cuda
Modalities: RNA, ATAC | label=Spatial_Label
Spatial graph: shape=(2248, 2248), nnz=8286
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/050 | total=23.338100 | rec=0.162293 | mgcl=7.725269 | cluster=3.435690 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.105e-02 | neg_count=5051256 | snf_masked_positions=0 | effC=15.978 | gradC=0.000e+00
epoch 002/050 | total=23.296076 | rec=0.150283 | mgcl=7.715264 | cluster=3.435288 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.443e-02 | neg_count=5051256 | snf_masked_positions=0 | effC=15.984 | gradC=0.000e+00
epoch 003/050 | total=23.260895 | rec=0.139457 | mgcl=7.707146 | cluster=3.434963 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.651e-02 | neg_count=5051256 | snf_masked_positions=0 | effC=15.988 | gradC=0.000e+00
epoch

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: S2-E18 | spots=2248 | device=cuda
Modalities: RNA, ATAC | label=Spatial_Label
Spatial graph: shape=(2248, 2248), nnz=8286
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/050 | total=23.333382 | rec=0.160937 | mgcl=7.724148 | cluster=3.435792 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.161e-02 | neg_count=5051256 | snf_masked_positions=0 | effC=15.978 | gradC=0.000e+00
epoch 002/050 | total=23.293091 | rec=0.148142 | mgcl=7.714983 | cluster=3.435352 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.496e-02 | neg_count=5051256 | snf_masked_positions=0 | effC=15.984 | gradC=0.000e+00
epoch 003/050 | total=23.258493 | rec=0.136740 | mgcl=7.707251 | cluster=3.434995 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=1.641e-02 | neg_count=5051256 | snf_masked_positions=0 | effC=15.988 | gradC=0.000e+00
epoch

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: S2-E18 | spots=2248 | device=cuda
Modalities: RNA, ATAC | label=Spatial_Label
Spatial graph: shape=(2248, 2248), nnz=8286
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/050 | total=23.329287 | rec=0.159029 | mgcl=7.723419 | cluster=3.435204 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=8.860e-03 | neg_count=5051256 | snf_masked_positions=0 | effC=15.986 | gradC=0.000e+00
epoch 002/050 | total=23.296541 | rec=0.146848 | mgcl=7.716564 | cluster=3.434880 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=8.225e-03 | neg_count=5051256 | snf_masked_positions=0 | effC=15.990 | gradC=0.000e+00
epoch 003/050 | total=23.272545 | rec=0.135765 | mgcl=7.712260 | cluster=3.434608 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=8.635e-03 | neg_count=5051256 | snf_masked_positions=0 | effC=15.993 | gradC=0.000e+00
epoch

## Cell 19：汇总 S2-E18 10-seed 正式结果与 BSRR 增益

In [19]:
s2e18_summary = summarize_dataset("s2e18")

print("\nPASS: S2-E18 summary complete.")

===== S2E18 per-seed results =====
seed 0: ARI=0.316088 | NMI=0.464585 | ΔARI=+0.000929 | ΔNMI=+0.011098
seed 1: ARI=0.332305 | NMI=0.472178 | ΔARI=+0.009528 | ΔNMI=+0.003803
seed 2: ARI=0.340047 | NMI=0.472118 | ΔARI=+0.002611 | ΔNMI=+0.000639
seed 3: ARI=0.309429 | NMI=0.470290 | ΔARI=-0.029118 | ΔNMI=+0.008404
seed 4: ARI=0.326862 | NMI=0.476339 | ΔARI=-0.010125 | ΔNMI=-0.000926
seed 5: ARI=0.334200 | NMI=0.466112 | ΔARI=-0.007651 | ΔNMI=-0.003602
seed 6: ARI=0.327509 | NMI=0.481915 | ΔARI=+0.014878 | ΔNMI=+0.014722
seed 7: ARI=0.310465 | NMI=0.462593 | ΔARI=+0.032678 | ΔNMI=+0.011695
seed 8: ARI=0.328827 | NMI=0.479946 | ΔARI=-0.000417 | ΔNMI=+0.001214
seed 9: ARI=0.318822 | NMI=0.480392 | ΔARI=-0.036153 | ΔNMI=-0.027147

===== S2E18 10-run summary =====
Official ARI = 0.324455 ± 0.009784
Official NMI = 0.472647 ± 0.006539

===== Raw concat-Z =====
Raw ARI = 0.326739 ± 0.020337
Raw NMI = 0.470657 ± 0.015000

===== BSRR effect =====
Mean ΔARI = -0.002284
Mean ΔNMI = +0.001990
ARI im

## Cell 20：统一汇总 5 个真实数据集的正式结果

In [20]:
hlna1_summary = summarize_dataset("hlna1")

all_summaries = [
    hlna1_summary,
    d1_summary,
    e185_summary,
    s2e15_summary,
    s2e18_summary,
]

print("\n" + "=" * 100)
print("FINAL 5-DATASET SUMMARY")
print("=" * 100)

print(
    f"{'Dataset':<10}"
    f"{'ARI mean±std':<24}"
    f"{'NMI mean±std':<24}"
    f"{'ΔARI':<12}"
    f"{'ΔNMI':<12}"
    f"{'ARI +':<8}"
    f"{'NMI +':<8}"
)

print("-" * 100)

for s in all_summaries:

    ari_text = (
        f"{s['ari_mean']:.4f} ± "
        f"{s['ari_std']:.4f}"
    )

    nmi_text = (
        f"{s['nmi_mean']:.4f} ± "
        f"{s['nmi_std']:.4f}"
    )

    print(
        f"{s['dataset']:<10}"
        f"{ari_text:<24}"
        f"{nmi_text:<24}"
        f"{s['delta_ari_mean']:+.4f}     "
        f"{s['delta_nmi_mean']:+.4f}     "
        f"{s['ari_positive_seeds']}/10    "
        f"{s['nmi_positive_seeds']}/10"
    )

print("\nPASS: all 5 datasets summarized.")

===== HLNA1 per-seed results =====
seed 0: ARI=0.239996 | NMI=0.336198 | ΔARI=-0.001411 | ΔNMI=-0.000117
seed 1: ARI=0.264916 | NMI=0.364676 | ΔARI=+0.003953 | ΔNMI=+0.006192
seed 2: ARI=0.209397 | NMI=0.337053 | ΔARI=-0.008539 | ΔNMI=-0.007710
seed 3: ARI=0.227045 | NMI=0.334193 | ΔARI=+0.000870 | ΔNMI=+0.003929
seed 4: ARI=0.231193 | NMI=0.315773 | ΔARI=-0.007959 | ΔNMI=-0.004218
seed 5: ARI=0.235218 | NMI=0.347538 | ΔARI=+0.001373 | ΔNMI=+0.004769
seed 6: ARI=0.260178 | NMI=0.338467 | ΔARI=-0.004283 | ΔNMI=+0.008827
seed 7: ARI=0.238806 | NMI=0.329467 | ΔARI=-0.004431 | ΔNMI=+0.002991
seed 8: ARI=0.229868 | NMI=0.323038 | ΔARI=+0.008818 | ΔNMI=+0.007357
seed 9: ARI=0.262099 | NMI=0.310668 | ΔARI=+0.008464 | ΔNMI=+0.005488

===== HLNA1 10-run summary =====
Official ARI = 0.239872 ± 0.016808
Official NMI = 0.333707 ± 0.014747

===== Raw concat-Z =====
Raw ARI = 0.240186 ± 0.015204
Raw NMI = 0.330956 ± 0.014686

===== BSRR effect =====
Mean ΔARI = -0.000315
Mean ΔNMI = +0.002751
ARI im

## Cell 21：保存 5 数据集正式汇总结果

In [21]:
import json
import csv
from pathlib import Path

summary_dir = Path(
    "/kaggle/working/SpaMGCL/SpaMGCL/formal_summary"
)

summary_dir.mkdir(
    parents=True,
    exist_ok=True,
)

json_path = summary_dir / "formal_5datasets_10seeds_50ep.json"

with json_path.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        all_summaries,
        f,
        indent=2,
        ensure_ascii=False,
    )

csv_path = summary_dir / "formal_5datasets_10seeds_50ep.csv"

fieldnames = [
    "dataset",
    "ari_mean",
    "ari_std",
    "nmi_mean",
    "nmi_std",
    "raw_ari_mean",
    "raw_nmi_mean",
    "delta_ari_mean",
    "delta_nmi_mean",
    "ari_positive_seeds",
    "nmi_positive_seeds",
]

with csv_path.open(
    "w",
    newline="",
    encoding="utf-8",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=fieldnames,
    )

    writer.writeheader()

    for row in all_summaries:
        writer.writerow(row)

print("Saved:")
print(json_path)
print(csv_path)

print("\nPASS: summary files saved.")

Saved:
/kaggle/working/SpaMGCL/SpaMGCL/formal_summary/formal_5datasets_10seeds_50ep.json
/kaggle/working/SpaMGCL/SpaMGCL/formal_summary/formal_5datasets_10seeds_50ep.csv

PASS: summary files saved.


## Cell 22：检查 50 个正式 run 是否全部完成

In [22]:
from pathlib import Path

result_root = Path(
    "/kaggle/working/SpaMGCL/SpaMGCL/results_clean"
)

dataset_keys = [
    "hlna1",
    "d1",
    "e185",
    "s2e15",
    "s2e18",
]

completed = []
missing = []

for dataset_key in dataset_keys:

    for seed in range(10):

        run_name = (
            f"{dataset_key}_formal_50ep_seed{seed}"
        )

        metrics_path = (
            result_root
            / run_name
            / "metrics.json"
        )

        if metrics_path.exists():
            completed.append(run_name)
        else:
            missing.append(run_name)

print("Completed:", len(completed))
print("Missing:", len(missing))

if missing:
    print("\nMissing runs:")
    for name in missing:
        print(" -", name)

assert len(completed) == 50
assert len(missing) == 0

print("\nPASS: 50 / 50 formal runs completed.")

Completed: 50
Missing: 0

PASS: 50 / 50 formal runs completed.


## Cell 23：打包全部正式实验结果

In [ ]:
import shutil
from pathlib import Path

project_root = Path(
    "/kaggle/working/SpaMGCL/SpaMGCL"
)

archive_root = Path(
    "/kaggle/working/SpaMGCL_formal_5datasets_10seeds_50ep"
)

zip_path = shutil.make_archive(
    str(archive_root),
    "zip",
    root_dir=project_root,
    base_dir="results_clean",
)

zip_path = Path(zip_path)

print("Archive:")
print(zip_path)

print("\nExists:", zip_path.exists())

print(
    "Size (MB):",
    round(
        zip_path.stat().st_size
        / 1024
        / 1024,
        2,
    ),
)

assert zip_path.exists()

print("\nPASS: formal results archived.")